In [53]:
import pandas as pd
import numpy as np

In [54]:
df = pd.read_csv('../../output/processed_data/01_sector_handled.csv')

# Mapping suffix

In [55]:
df['suffix'].value_counts()

suffix
N0000    65606
X0000     4807
U0000      407
P0000      224
Name: count, dtype: int64

Removing records with suffixes U0000 & P0000      

In [56]:
#getting indexes
idx = np.array(df[(df['suffix'] == 'P0000' )| (df['suffix'] == 'U0000')].index.values)

print(f'Number of records before deletion : {df.shape[0]}')
df.drop(idx,axis = 0,inplace = True)
print(f'Number of records after deletion : {df.shape[0]}')

Number of records before deletion : 71044
Number of records after deletion : 70413


In [57]:
df.head()

,symbol,date,open,high,low,close,volume,base_symbol,suffix,sector
0,ABAN.N0000,2025-02-03,468.0,485.0,447.50,470.00,10154.0,ABAN,N0000,Consumer Durables
1,ABAN.N0000,2025-02-05,460.0,460.0,430.00,433.00,14587.0,ABAN,N0000,Consumer Durables
2,ABAN.N0000,2025-02-06,437.0,475.0,430.00,440.75,5961.0,ABAN,N0000,Consumer Durables
3,ABAN.N0000,2025-02-07,455.0,470.0,451.00,465.00,2629.0,ABAN,N0000,Consumer Durables
4,ABAN.N0000,2025-02-10,455.0,468.0,441.75,468.00,1763.0,ABAN,N0000,Consumer Durables


In [58]:

# 1. Remove specific outliers (Targeted ticker + suffix + date combinations)
outlier_records = [
    ('COLO', 'N0000', '2025-10-08'),
    ('CIC', 'X0000', '2025-10-07')
]
for ticker, suf, date in outlier_records:
    df = df[~((df['symbol'] == ticker) & (df['suffix'] == suf) & (df['date'] == date))]


penny_stocks = [
    'SEMB.N0000', 'SEMB.X0000', 'CSF.N0000', 'ASPH.N0000', 
    'TESS.N0000', 'TESS.X0000', 'UBF.N0000', 'ACME.N0000', 'MULL.N0000'
]

#
low_liquidity_stocks = [
    'SLND.N0000', 'HARI.N0000', 'BREW.N0000', 'HUNT.N0000', 'LAMB.N0000', 
    'CINS.N0000', 'CINS.X0000', 'CTEA.N0000', 'LPRT.N0000', 'NEH.N0000', 
    'OFEQ.N0000', 'SFCL.N0000', 'CPRT.N0000', 'NTB.X0000', 'KFP.N0000', 
    'MSL.N0000', 'LION.N0000', 'PARA.N0000', 'SIL.N0000', 'CTHR.N0000'
]
low_volume_stocks = [
    'UCAR.N0000', 'SMOT.N0000', 'AUTO.N0000', 'MAL.X0000', 'CERA.N0000', 
    'AAF.N0000', 'BUKI.N0000', 'CARG.N0000', 'CDB.N0000', 'ASHO.N0000', 
    'RENU.N0000', 'RGEM.N0000', 'MAL.N0000', 'CIT.N0000', 'JKL.N0000', 
    'CARS.N0000', 'COMD.N0000', 'ATLL.N0000', 'REXP.N0000', 'RFL.N0000',
    'TANG.N0000', 'DIMO.N0000', 'AINS.N0000', 'HAPU.N0000', 'ABAN.N0000',
    'GEST.N0000', 'BRR.N0000', 'RCH.N0000', 'RPBH.N0000', 'CFI.N0000',
    'AFSL.N0000', 'UDPL.N0000', 'MRH.N0000', 'AGST.X0000', 'LHL.N0000',
    'KGAL.N0000', 'CALF.N0000', 'SING.N0000', 'CTC.N0000', 'PALM.N0000',
    'TAJ.N0000', 'TRAN.N0000', 'PEG.N0000', 'ODEL.N0000', 'SHAW.N0000',
    'ONAL.N0000', 'CABO.N0000', 'CHMX.X0000','CHL.X0000','CDB.X0000'
]
# Create a combined mask for symbols to remove
combined_id = df['symbol'] + '.' + df['suffix']
df = df[~combined_id.isin(penny_stocks + low_liquidity_stocks + low_volume_stocks )].copy()

print(f"Post-cleaning: {df['symbol'].nunique()} instruments and {len(df)} records remain.")

Post-cleaning: 286 instruments and 70413 records remain.


In [59]:
def map_suffix(suf):
    if suf == 'N0000':  # Ordinary voting shares
        return 0
    if suf == 'X0000':  # Non-voting shares
        return 1
    return -1           # Unknown / other

df['suffix_cat'] = df['suffix']
df['suffix_cat'] = df['suffix_cat'].map(map_suffix)


In [60]:
df['suffix_cat'].value_counts()

suffix_cat
0    65606
1     4807
Name: count, dtype: int64

In [61]:
df.drop('suffix',axis = 1,inplace = True)

Deletion of delisted stocks

##### Browns beach hotels -BBH | Nation Lanka Finance - CSF

In [62]:
print(f'Number of records before deletion : {df.shape[0]}')
df.drop(df[df['symbol'] == 'BBH'].index,inplace = True)
df.drop(df[df['symbol'] == 'CSF'].index,inplace = True)
print(f'Number of records after deletion : {df.shape[0]}')

Number of records before deletion : 70413
Number of records after deletion : 70413


Also deletion of INME due to lack of data records within the period


In [63]:
print(f'Number of records before deletion : {df.shape[0]}')
df.drop(df[df['symbol'] == 'INME'].index,inplace = True)
print(f'Number of records after deletion : {df.shape[0]}')

Number of records before deletion : 70413
Number of records after deletion : 70413


In [64]:
df.head()

,symbol,date,open,high,low,close,volume,base_symbol,sector,suffix_cat
0,ABAN.N0000,2025-02-03,468.0,485.0,447.50,470.00,10154.0,ABAN,Consumer Durables,0
1,ABAN.N0000,2025-02-05,460.0,460.0,430.00,433.00,14587.0,ABAN,Consumer Durables,0
2,ABAN.N0000,2025-02-06,437.0,475.0,430.00,440.75,5961.0,ABAN,Consumer Durables,0
3,ABAN.N0000,2025-02-07,455.0,470.0,451.00,465.00,2629.0,ABAN,Consumer Durables,0
4,ABAN.N0000,2025-02-10,455.0,468.0,441.75,468.00,1763.0,ABAN,Consumer Durables,0


In [65]:
df.to_csv('../../output/processed_data/02_suffix_handled.csv',index = False)